In [1]:
###--------------------------------------------LOAD LIBRARIES
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import copy
import os, glob
import random


paths = glob.glob('../../processed_data/00-prepare_GO/01-fetch_genes_for_GO_pathways/*.csv')                         # read gene per pathway data

In [61]:
print('Reading anndata')
adata =  sc.read("/home/MRMelissaGrantPeters/MinaRyten/Melissa/spatialDLPFC/nonIF_c2l_anndata_combined_30samples.h5ad")

# Prepare data
subset = adata[adata.obs['bayesSpace_harmony_7'].isin([1,2,3,4,5,6,7])]   # Excluding WM and meningeal layers

subset.obs['bayesSpace_harmony_7'] = subset.obs['bayesSpace_harmony_7'].replace({   # Renaming to match order of MRI (L1, closer to WM, L4 closer to meninges)
                                                     1 : 'SpD1',
                                                     2 : 'SpD1', 
                                                     3 : 'SpD2', 
                                                     4 : 'SpD3', 
                                                     5 : 'WM',
                                                     6 : 'SpD4',
                                                     7 : 'WM'})

subset.obs['sample_id'] = subset.obs['sample_id'].astype(str)

df_deconvo = subset.obs[np.isnan(subset.obs['Oligo'])==False]

Reading anndata


/tmp/ipykernel_2501824/815679380.py:7: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  subset.obs['bayesSpace_harmony_7'] = subset.obs['bayesSpace_harmony_7'].replace({   # Renaming to match order of MRI (L1, closer to WM, L4 closer to meninges)


In [98]:
full_pathway_list = []

for m in range(0,5):
    print('Reading data')
    search_paths = pd.read_csv('../../raw_data/keyword_search_amiGO/%s_GO_terms.csv' % paths[0].split('_')[7])
    df = pd.read_csv(paths[m], index_col=0)  # Fetch gene and pathway information for a category
    pathways = list(df['GO'].unique())
    #pathways = list(set(df['GO'].unique()) & set(search_paths['GO_term']))  # Fetch the pathway names
    categories = pd.DataFrame(paths)[0].str.split('/', expand = True)[5].str.split('.csv', expand = True)[0] #  Fetch category names
    print(len(pathways))
    print('Start pathway scoring')
    genes_per_pathway = []
    five_and_more = []
    for n in range(0,len(pathways)):
        print(categories[m], pathways[n])
        genes = list(df[df['GO']==pathways[n]]['SYMBOL'])  #list genes
        genes_per_pathway.append(len(genes))
        captured_in_data = list(set(genes) & set(adata.var['gene_name']))
        if (len(genes)>4 and (len(captured_in_data)>(0.5*len(genes)))):  # Filter pathways so they need to have at least 5 genes, of which at least 50% are captured in the DLPFC data
            five_and_more.append(categories[m]+'_'+pathways[n])
            full_pathway_list.append(categories[m]+'_'+pathways[n])
        else:
            print('NOPE')
    pd.DataFrame(five_and_more, columns = ['pathways_with_min_5_genes']).to_csv('../../processed_data/01b-4layers_withendo/paths_to_keep_%s.csv' % (categories[m]))

final = pd.DataFrame(full_pathway_list, columns = ['pathways_with_min_5_genes'])
final["pathways_with_min_5_genes"] = final["pathways_with_min_5_genes"].str.replace("genes_", "", regex=False).str.replace("GO_terms_", "", regex=False).str.replace(":", "", regex=False)
pd.DataFrame(final, columns = ['pathways_with_min_5_genes']).to_csv('../../processed_data/01b-4layers_withendo/paths_to_keep_all.csv')

Reading data
145
Start pathway scoring
genes_inflammation_GO_terms GO:0030851
genes_inflammation_GO_terms GO:0030221
NOPE
genes_inflammation_GO_terms GO:0030222
NOPE
genes_inflammation_GO_terms GO:0030223
genes_inflammation_GO_terms GO:0030853
genes_inflammation_GO_terms GO:0030854
genes_inflammation_GO_terms GO:0035854
NOPE
genes_inflammation_GO_terms GO:0045645
NOPE
genes_inflammation_GO_terms GO:0045659
NOPE
genes_inflammation_GO_terms GO:0045660
NOPE
genes_inflammation_GO_terms GO:0032680
genes_inflammation_GO_terms GO:0032720
NOPE
genes_inflammation_GO_terms GO:0032760
genes_inflammation_GO_terms GO:0038066
genes_inflammation_GO_terms GO:1900744
genes_inflammation_GO_terms GO:1900745
genes_inflammation_GO_terms GO:1903753
genes_inflammation_GO_terms GO:0072538
genes_inflammation_GO_terms GO:0072539
NOPE
genes_inflammation_GO_terms GO:0072540
NOPE
genes_inflammation_GO_terms GO:2000316
NOPE
genes_inflammation_GO_terms GO:2000317
NOPE
genes_inflammation_GO_terms GO:2000318
NOPE
gene

In [99]:
final

,pathways_with_min_5_genes
0,inflammation_GO0030851
1,inflammation_GO0030223
2,inflammation_GO0030853
3,inflammation_GO0030854
4,inflammation_GO0032680
...,...
1298,myelin_GO0004707
1299,myelin_GO0048709
1300,myelin_GO0048713
1301,myelin_GO0048714


In [95]:
(len(captured_in_data))

11